[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/14-cost-latency.ipynb)

## Colab setup

Run the next cell first on Google Colab. It sparse-checkouts this public repo and installs packages only when the kernel is Colab. Local Jupyter and VS Code skip that work.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) or in an environment cell: `os.environ["OPENAI_API_KEY"] = "sk-..."`. Do that before the lesson setup cell. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

In [ ]:
# Colab setup. Run this cell before the other code cells.
# A fresh Colab runtime cannot import tutorial.common yet, so this cell
# sparse-checkouts the repo first. Local Jupyter and VS Code skip that.
#
# An API key is required. On Google Colab, set OPENAI_API_KEY in one of these ways:
#   * Secrets (the key icon): a secret named OPENAI_API_KEY
#   * an environment cell: os.environ["OPENAI_API_KEY"] = "sk-..."
# Locally, put the key in the repository-root .env.
# Do not commit a key. This cell does not print the value.
# Edits you make in Colab stay in the session. They do not push to GitHub.

import sys
from pathlib import Path


def _on_colab():
    try:
        import google.colab
    except ImportError:
        return False
    return Path("/content").is_dir() and google.colab is not None


def _use(root):
    if not (Path(root) / "tutorial" / "common" / "colab.py").is_file():
        return False
    text = str(root)
    if text not in sys.path:
        sys.path.insert(0, text)
    return True


ready = False
if _on_colab():
    repo = Path("/content/book-agents")
    if not _use(repo):
        import shutil
        import subprocess

        if repo.exists() and not (repo / ".git").exists():
            raise RuntimeError(
                str(repo) + " exists but is not a git checkout. "
                "Move that folder aside and run this cell again."
            )
        if repo.exists():
            shutil.rmtree(repo)
        url = "https://github.com/junlinghu/book-agents.git"
        try:
            subprocess.check_call([
                "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                url, str(repo),
            ])
            subprocess.check_call([
                "git", "-C", str(repo), "sparse-checkout", "set", "tutorial",
            ])
        except subprocess.CalledProcessError:
            if repo.exists():
                shutil.rmtree(repo)
            subprocess.check_call(["git", "clone", "--depth", "1", url, str(repo)])
        if not _use(repo):
            raise RuntimeError(
                "Colab setup could not find tutorial/common/colab.py after cloning."
            )
    ready = True
else:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if _use(candidate):
            ready = True
            break
    if not ready:
        print("Not Colab. Skipped clone and pip install.")

if ready:
    from tutorial.common.colab import setup_colab

    setup_colab()


# 14. Cost and latency

A loop that hides its token count will surprise you on the second week. This notebook logs tokens, latency, and an illustrative cost on every model call.

It also caches read-only tools. The hours question reads `faq.md` twice. The second read should say `cache: HIT`. That second call is also the repeat nudge from tutorial 3. The cache still serves the file. A third identical call would stop the loop.

`route_task` prints whether the question looks like a short lookup or a shop decision. Both routes name `gpt-4.1-mini` here. Swapping the id is a change to `MODEL` in `tutorial/common/client.py`. The trace from tutorial 13 is imported, so spans are still recorded.

Shared helpers this lesson needs are imported, not copied from earlier notebooks. You can run this file by itself.

## Setup

This notebook calls the OpenAI Chat Completions API. `OPENAI_API_KEY` is required. Locally the key is loaded from the repository-root `.env` by `tutorial/common/client.py`. On Colab, the setup cell above reads a secret of the same name. The value is not printed. A missing key stops the run.

Companion notes: `14-cost-latency.md`.

In [ ]:
import sys
from pathlib import Path


def find_root():
    """Repo root, whether the kernel started here or in tutorial."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        marker = candidate / "tutorial" / "common" / "client.py"
        if marker.is_file():
            return candidate
    raise RuntimeError(
        "Open this notebook inside the book-agents repository. "
        "The setup cell looks for tutorial/common/client.py."
    )


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.common.client import describe_runtime
from tutorial.runtime import chat, require_key

LESSON = '14-cost-latency'
LESSON_NUMBER = 14
require_key()
print("tutorial:", LESSON)
print(describe_runtime())


## Shared pieces

These imports are the earlier pieces this lesson uses. They come from `tutorial.common` and `tutorial.runtime`. You do not need to run the previous notebook first.


In [ ]:
import json

import tutorial.common.trace  # installs span hooks on the loop
from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent
from tutorial.common.shelf import reset_db


## Tutorial 14 — cost, latency, and a cache

Token totals and an illustrative cost. A second read of the same document can hit the cache.

In [ ]:
# Illustrative rates for this lab's ledger, in USD per million tokens.
# They are not an invoice. Check current OpenAI pricing before you budget with them.
import json

from tutorial.common.harness import call_tool, set_hook

INPUT_USD_PER_MILLION = 0.40
OUTPUT_USD_PER_MILLION = 1.60

CACHEABLE = {
    "get_shop_fact",
    "read_shop_file",
    "query_inventory",
    "list_notes",
    "read_note",
    "load_skill",
    "fetch_page",
    "memory_search",
    "read_supplier_note",
}
CACHE = {}
CACHE_EVENTS = []


def reset_cache():
    CACHE.clear()
    CACHE_EVENTS.clear()


def route_task(user_text):
    """Pick a model class. This lab still calls one model; the log shows the decision."""
    text = user_text.lower()
    if any(word in text for word in ("restock", "ticket", "verify")):
        return {
            "task": "shop-decision",
            "model": "gpt-4.1-mini",
            "reason": "needs tools and a policy check",
        }
    return {
        "task": "counter-question",
        "model": "gpt-4.1-mini",
        "reason": "short lookup on the default tool-calling model",
    }


def cost_summary(usage_rows):
    prompt = sum(row["prompt_tokens"] for row in usage_rows)
    completion = sum(row["completion_tokens"] for row in usage_rows)
    usd = (prompt * INPUT_USD_PER_MILLION + completion * OUTPUT_USD_PER_MILLION) / 1_000_000
    latency = sum(row["latency_ms"] for row in usage_rows)
    return {
        "prompt_tokens": prompt,
        "completion_tokens": completion,
        "estimated_usd": round(usd, 6),
        "latency_ms": round(latency, 3),
        "rate_note": "illustrative, not an invoice",
    }


def cached_call(call):
    """Return a cached read, or run the tool and store it."""
    name = call["name"]
    if name not in CACHEABLE:
        return call_tool(call), False
    key = name + " " + json.dumps(call["arguments"], sort_keys=True, default=str)
    if key in CACHE:
        print("cache: HIT", key)
        CACHE_EVENTS.append("HIT " + key)
        return CACHE[key], True
    print("cache: MISS", name)
    CACHE_EVENTS.append("MISS " + name)
    result = call_tool(call)
    CACHE[key] = result
    return result, False


set_hook("route_task", route_task)
set_hook("cached_call", cached_call)


## Run

`system_text()` mentions only the tools imported above. Run this cell after the ones above. Each model turn calls the Chat Completions API. A check prints a warning when the wording differs from the expected trace, and the notebook continues.

In [ ]:
reset_db()
reset_cache()
question = "What are the café hours?"
print("route preview:", json.dumps(route_task(question)))
result = run_agent(question, system_text(), trace_id="hours-cached")
print("ANSWER:", result["text"])
summary = cost_summary(result["usage"])
print("ledger:", json.dumps(summary))
print("cache events:", CACHE_EVENTS)
check(result["stopped"] == "final", "hours question finished")
check(any(event.startswith("MISS") for event in CACHE_EVENTS), "the first read missed the cache")
check(any(event.startswith("HIT") for event in CACHE_EVENTS), "the second read hit the cache")
check(summary["prompt_tokens"] > 0, "the ledger counted prompt tokens")
check(summary["estimated_usd"] >= 0, "the ledger has an illustrative cost")
check("closed monday" in result["text"].lower() or "Closed Monday" in result["text"], "hours come from the FAQ")
check(len(result["spans"]) >= 2, "spans from tutorial 13 are still recorded")
